# {{PROJECT_NAME}}

KFP v2 fine-tuning pipeline on the Miramar platform.

**Development workflow:**
1. Write step logic in the `@dsl.component` cells below
2. Wire steps in the **Pipeline** cell
3. Save (`Ctrl+S`), then run **Build → pipeline.py**
4. Compile and submit in **Compile & Submit**

## Step Development

Write your pipeline step logic directly inside each `@dsl.component` function body.

- **Imports must be inside the function** — KFP runs each component in an isolated container
- Set `base_image` and `packages_to_install` on `@dsl.component` to match your step’s runtime
- Rename functions to match your steps — the Pipeline cell and extraction follow function names

In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Artifact

### Step 1

In [ ]:
@dsl.component(base_image="python:3.11-slim")
def step_1(output_data: Output[Dataset]):
    # All imports inside the function body
    import pathlib
    pathlib.Path(output_data.path).write_text("placeholder")
    print("step_1 done")

### Step 2

In [ ]:
@dsl.component(base_image="python:3.11-slim")
def step_2(input_data: Input[Dataset], output_data: Output[Dataset]):
    import pathlib
    data = pathlib.Path(input_data.path).read_text()
    pathlib.Path(output_data.path).write_text(data)
    print("step_2 done")

### Step 3

In [ ]:
@dsl.component(base_image="python:3.11-slim")
def step_3(input_data: Input[Dataset], output_data: Output[Dataset]):
    import pathlib
    data = pathlib.Path(input_data.path).read_text()
    pathlib.Path(output_data.path).write_text(data)
    print("step_3 done")

### Step 4

In [ ]:
@dsl.component(base_image="python:3.11-slim")
def step_4(input_data: Input[Dataset], output_model: Output[Model]):
    import pathlib
    data = pathlib.Path(input_data.path).read_text()
    pathlib.Path(output_model.path).write_text(data)
    print("step_4 done")

### Step 5

In [ ]:
@dsl.component(base_image="python:3.11-slim")
def step_5(input_model: Input[Model], output_model: Output[Model]):
    import pathlib
    data = pathlib.Path(input_model.path).read_text()
    pathlib.Path(output_model.path).write_text(data)
    print("step_5 done")

### Step 6

In [ ]:
@dsl.component(base_image="python:3.11-slim")
def step_6(input_model: Input[Model], output_artifact: Output[Artifact]):
    import pathlib
    pathlib.Path(output_artifact.path).write_text("placeholder")
    print("step_6 done")

### Step 7

In [ ]:
@dsl.component(base_image="python:3.11-slim")
def step_7(input_artifact: Input[Artifact]):
    import pathlib
    print(f"step_7: received artifact at {input_artifact.path}")
    print("step_7 done")

### Pipeline

Wire the step tasks together. Access outputs via `.output` (single unnamed output) or `.outputs["name"]` (named outputs).

In [ ]:
@dsl.pipeline(name="{{PROJECT_NAME}}")
def pipeline():
    t1 = step_1()
    t2 = step_2(input_data=t1.outputs["output_data"])
    t3 = step_3(input_data=t2.outputs["output_data"])
    t4 = step_4(input_data=t3.outputs["output_data"])
    t5 = step_5(input_model=t4.outputs["output_model"])
    t6 = step_6(input_model=t5.outputs["output_model"])
    step_7(input_artifact=t6.outputs["output_artifact"])

## Build → `pipeline.py`

Save the notebook first (`Ctrl+S`), then run this cell.

In [ ]:
import json, pathlib, re

def build_pipeline(notebook_path="notebook.ipynb"):
    nb = json.loads(pathlib.Path(notebook_path).read_text())
    step_srcs, pipeline_src = [], None
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        tags = cell.get("metadata", {}).get("tags", [])
        src = "".join(cell["source"])
        if "kfp_step" in tags:
            step_srcs.append(src)
        elif "kfp_pipeline" in tags:
            pipeline_src = src
    if not step_srcs:
        raise RuntimeError("No cells tagged 'kfp_step' found.")
    if pipeline_src is None:
        raise RuntimeError("No cell tagged 'kfp_pipeline' found.")
    names = [m.group(1) for src in step_srcs
             for m in [re.search(r"^def (\w+)\(", src, re.MULTILINE)] if m]
    out  = "# Generated by notebook.ipynb \u2014 do not edit manually.\n"
    out += "# Re-run the Build cell to regenerate.\n\n"
    out += "from kfp import dsl\n"
    out += "from kfp.dsl import Input, Output, Dataset, Model, Artifact\n\n\n"
    out += "\n\n\n".join(step_srcs)
    out += "\n\n\n"
    out += pipeline_src
    out += "\n"
    pathlib.Path("pipeline.py").write_text(out)
    print(f"Wrote pipeline.py \u2014 {len(names)} component(s): {', '.join(names)}")

build_pipeline()

## Compile & Submit

In [ ]:
from kfp import compiler
from pipeline import pipeline
compiler.Compiler().compile(pipeline_func=pipeline, package_path='/tmp/pipeline.yaml')
print('Compiled → /tmp/pipeline.yaml')

In [ ]:
# Connect to KFP (requires SSH tunnel: ssh -L 8080:localhost:8080 spark-79b7.local)
import kfp
client = kfp.Client(host='http://localhost:8080')
client.list_pipelines()

In [ ]:
run = client.create_run_from_pipeline_package(
    pipeline_file='/tmp/pipeline.yaml',
    arguments={},
    run_name='notebook-run',
)
print(f'Run ID: {run.run_id}')
print(f'UI: http://localhost:8080/#/runs/details/{run.run_id}')

In [ ]:
import time
run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.state
    print(f'  {state}')
    if state in ('SUCCEEDED', 'FAILED', 'CANCELED'):
        break
    time.sleep(10)